## The networking mental model

When a container starts on a normal (non-`host`) network, the kernel gives it a **network namespace** of its own — the module-01 isolation primitive, applied to networking. That namespace has:

- Its own interfaces (usually just loopback `lo` and a virtual `eth0`)
- Its own IP address(es)
- Its own routing table
- Its own `iptables` / `nftables` rules
- Its own set of open ports

From **inside**, this looks like a tiny isolated machine on a network. From the **host**, the container's `eth0` is one end of a **virtual ethernet pair** (`veth`); the other end plugs into a Linux **bridge** — `docker0` for the default network, or a `br-xxxx` for a user-defined one.

```
   Container A                 Host
   +-----------+  veth pair    ┌───────────────────────────┐
   | eth0      |===============│ docker0 bridge 172.17.0.1 │
   | 172.17.0.2|               │        │        │         │
   +-----------+               │      (NAT to eth0 → LAN)  │
                               └───────────────────────────┘
```

Two consequences that explain almost everything later:

- **Containers on the same bridge talk directly** on their container ports — the bridge is a virtual switch between them (that's why same-network containers need no `-p`).
- **Outbound traffic is NAT'd** through the host's real interface, so a container can reach the internet — but **inbound is blocked** by default, because the bridge is private. Getting traffic *in* requires publishing a port (section 7).

Everything in this module is a variation on this one picture: namespaces on a bridge, NAT out, publish to come in.